# 10 – Systematic experiments, analysis and method-selection guide
`QUICK_MODE = True` for development; set `False` for the final run (long). Results are appended to `results/experiment_results.csv` (resumable).

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
root = next(p for p in [Path.cwd(), Path.cwd().parent] if (p / "src").exists())
sys.path.insert(0, str(root))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
try:
    from IPython.display import display, Image
except ImportError:
    display = print
PARSED = root / "data" / "processed" / "gas_sensor_drift_parsed.csv"
USE_STANDIN = False   # True ONLY to test the notebook without UCI data (results then are NOT valid)
if USE_STANDIN:
    from src.synthetic import make_standin
    df = make_standin(5000)
    print("!! STAND-IN DATA - not UCI")
else:
    assert PARSED.exists(), "Run notebook 01 (or `python -m src.data_loader`) first"
    df = pd.read_csv(PARSED)
FIG = root / "results" / "figures"; TAB = root / "results" / "tables"
FIG.mkdir(parents=True, exist_ok=True); TAB.mkdir(parents=True, exist_ok=True)
print(df.shape)

In [ ]:
from src.experiments import build_main_conditions, build_extra_conditions, run_experiments
from src.registry import ALGORITHMS
QUICK_MODE = True
DESIGN = 'factorial' if QUICK_MODE else 'ofat'      # full factorial is optional: DESIGN='factorial', MAX_CONDITIONS=...
N_SAMPLES = 800 if QUICK_MODE else 3000
SEEDS = list(range(1 if QUICK_MODE else 5))
MAX_CONDITIONS = None
RUN_EXTRAS = True
N_JOBS = 1
conds = build_main_conditions(QUICK_MODE, DESIGN, N_SAMPLES, SEEDS, MAX_CONDITIONS)
if RUN_EXTRAS: conds += build_extra_conditions(QUICK_MODE, N_SAMPLES, SEEDS)
print(len(conds), 'conditions x', len(ALGORITHMS), 'algorithms')
OUT_CSV = root / 'results' / 'experiment_results.csv'
res = run_experiments(df, conds, list(ALGORITHMS), OUT_CSV, n_jobs=N_JOBS)
res.groupby('algorithm')[['roc_auc','pr_auc','f1']].mean().round(3)

In [ ]:
from src.analysis import run_analysis, representative_figures, guide_to_markdown
res_dict = run_analysis(OUT_CSV, root / 'results', df, standin=USE_STANDIN)
display(res_dict['win_rates']); print(res_dict['friedman'])
print(guide_to_markdown(res_dict['guide']))

In [ ]:
representative_figures(df, FIG)   # ROC, PR, confusion matrices, PCA/t-SNE/UMAP, parallel coordinates

In [ ]:
for f in ['perf_vs_dimensionality_roc_auc','perf_vs_noise_roc_auc','perf_vs_anomaly_ratio_roc_auc','perf_vs_sensor_dropout_roc_auc','heatmap_algorithm_x_dimensionality','roc_point']:
    p = FIG / f'{f}.png'
    if p.exists(): print(f); display(Image(str(p)))

The generated summary is in `results/results_summary.md`; the report's Results/Discussion sections must be written from these tables.

### Result log (fill in after running)
*Not yet executed.* Copy the numbers printed above into your report; do not quote numbers from memory.